In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("IMDB Dataset.csv",nrows=1000)
data.head()
data["target"] = data["sentiment"].replace({
    "positive": 1,
    "negative": 0
})

C:\Users\mudha\AppData\Local\Temp\ipykernel_17200\3416658593.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["target"] = data["sentiment"].replace({


In [2]:
from transformers import AutoTokenizer

text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def convert_text_to_ids(text):
    encoded = text_tokenizer.encode(
        text,
        max_length=128,
        padding="max_length",
        truncation=True
    )
    return encoded

data["encoded_review"] = data["review"].apply(convert_text_to_ids)
data["encoded_review"].head()

0    [101, 2028, 1997, 1996, 2060, 15814, 2038, 385...
1    [101, 1037, 6919, 2210, 2537, 1012, 1026, 7987...
2    [101, 1045, 2245, 2023, 2001, 1037, 6919, 2126...
3    [101, 10468, 2045, 1005, 1055, 1037, 2155, 207...
4    [101, 9004, 3334, 4717, 7416, 1005, 1055, 1000...
Name: encoded_review, dtype: object

In [3]:
from sklearn.model_selection import train_test_split

features = data["encoded_review"]
labels = data["target"]

x_train, x_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42
)

In [4]:
from transformers import AutoModelForSequenceClassification
import torch

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
test_inputs = torch.tensor(list(x_test))

with torch.no_grad():
    result = sentiment_model(input_ids=test_inputs)

predictions = torch.argmax(result.logits, dim=1)

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc = accuracy_score(y_test, predictions)
prec = precision_score(y_test, predictions)
rec = recall_score(y_test, predictions)
f1_val = f1_score(y_test, predictions)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1_val)

Accuracy: 0.515
Precision: 0.48717948717948717
Recall: 0.19791666666666666
F1 Score: 0.2814814814814815
